In [1]:
from google.colab import drive
import os

# mount drive
drive.mount('/content/drive')

# clone repo
!git clone https://github.com/tum-ai/physicsnemo.git

%cd physicsnemo

# install requirements
%pip install -e .
%pip install -r examples/structural_mechanics/crash/requirements.txt

Mounted at /content/drive
Cloning into 'physicsnemo'...
remote: Enumerating objects: 23452, done.
remote: Counting objects: 100% (379/379), done.
remote: Compressing objects: 100% (251/251), done.
remote: Total 23452 (delta 240), reused 128 (delta 128), pack-reused 23073 (from 3)
Receiving objects: 100% (23452/23452), 282.77 MiB | 49.56 MiB/s, done.
Resolving deltas: 100% (14658/14658), done.
/content/physicsnemo
Obtaining file:///content/physicsnemo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 48.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.2/221.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 9.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.9/272.9 kB 17.6 MB/s eta 0:00:00


In [34]:
%cd ..

/content/physicsnemo


In [2]:
# Create the local simulations directory first
!mkdir -p examples/structural_mechanics/crash/simulations

# Link the contents of the simulations folder and copy the master CSV from Google Drive to local workspace
!ln -sf /content/drive/MyDrive/physicsnemo/simulations/* examples/structural_mechanics/crash/simulations/
!cp /content/drive/MyDrive/physicsnemo/simulations/bumper_beam_master_with_split.csv examples/structural_mechanics/crash/bumper_beam_master_with_split.csv

# Define Google Drive directories for keeping checkpoints, logs, and precomputed dataset statistics
drive_ckpt_dir = '/content/drive/MyDrive/physicsnemo/checkpoints'
drive_tb_dir = '/content/drive/MyDrive/physicsnemo/tensorboard_logs'
drive_stats_dir = '/content/drive/MyDrive/physicsnemo/stats'

# Create these folders in Google Drive if they do not exist
os.makedirs(drive_ckpt_dir, exist_ok=True)
os.makedirs(drive_tb_dir, exist_ok=True)
os.makedirs(drive_stats_dir, exist_ok=True)

# Navigate to the crash example directory
%cd examples/structural_mechanics/crash

# Build the global-features JSON
!python make_global_features.py



/content/physicsnemo/examples/structural_mechanics/crash
Wrote 14742 runs -> ./global_features.json  (keys: ['velocity_x', 'thickness_scale', 'rwall_origin_y'])


In [7]:
raw_data_dir = "/content/physicsnemo/examples/structural_mechanics/crash/simulations"
global_features = "/content/physicsnemo/examples/structural_mechanics/crash/global_features.json"

master_csv = "/content/physicsnemo/examples/structural_mechanics/crash/simulationsbumper_beam_master_with_split.csv"



In [ ]:
!python make_global_features.py

In [8]:
# Start training with the bumper_vtkhdf_geotransolver_oneshot config, redirecting outputs to Drive
!HYDRA_FULL_ERROR=1 python train.py --config-name=bumper_vtkhdf_geotransolver_oneshot \
    training.raw_data_dir={raw_data_dir} \
    training.raw_data_dir_validation={raw_data_dir} \
    training.global_features_filepath={global_features} \
    inference.raw_data_dir_test={raw_data_dir} \
    reader.master_csv={master_csv} \
    training.ckpt_path={drive_ckpt_dir} \
    training.tensorboard_log_dir={drive_tb_dir} \
    datapipe.stats_dir={drive_stats_dir} \
    training.num_training_samples=20 \
    training.num_validation_samples=3

2026-06-15 12:57:28.829368: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/content/physicsnemo/physicsnemo/distributed/manager.py:415: UserWarning: Could not initialize using ENV, SLURM or OPENMPI methods. Assuming this is a single process job
  warn(
[2026-06-15 12:57:40,291][main][INFO] - Config:
reader:
  _target_: vtkhdf_reader.Reader
  _convert_: all
  master_csv: /content/physicsnemo/examples/structural_mechanics/crash/simulationsbumper_beam_master_with_split.csv
  exclude_parts: []
datapipe:
  _target_: datapipe.CrashPointCloudDataset
  _convert_: all
  data_dir: /content/physicsnemo/examples/structural_mechanics/crash/simulations
  global_features_filepath: /content/physicsnemo/examples/structural_mechanics/crash/global_features.json
  